## Movie Recommender Comparison

### Load & Split Data

In [1]:
from data_preprocessing import load_data, leave_one_out_split

ratings, movies = load_data()

train_df, test_df = leave_one_out_split(ratings)

print(train_df.head())
print(test_df.head())

       userId  movieId  rating   timestamp
10538      68     1101     3.5  1158533140
50207     325       80     4.0  1039397611
10484      68      628     3.5  1158534567
1574       16      293     4.0  1377477870
70108     448    99992     3.0  1374953846
       userId  movieId  rating   timestamp
67037     432    77866     4.5  1335139641
42175     288      474     3.0   978465565
93850     599     4351     3.0  1498524542
6187       42     2987     4.0   996262677
12229      75     1610     4.0  1158989841


## Build, Recommend, and Eval Popularity Baseline

In [2]:
import pandas as pd

from models import build_popularity_baseline, recommend_popular_movies
from evaluate import evaluate_recommender

popularity_rankings = build_popularity_baseline(train_df, min_ratings=5)

popular_recommend_func = lambda user_id, k: recommend_popular_movies(
    user_id=user_id,
    train_df=train_df,
    popularity_df=popularity_rankings,
    n=k
)

popularity_results = evaluate_recommender(
    train_df=train_df,
    test_df=test_df,
    recommend_func=popular_recommend_func,
    model_name="Popularity Baseline",
    k=10
)

results_df = pd.DataFrame([popularity_results])
print(results_df)

def show_recommendations(movie_ids, movies_df):
    return movies_df[
        movies_df["movieId"].isin(movie_ids)
    ][["movieId", "title", "genres"]]

user_id = ratings["userId"].iloc[0]

rec_ids = recommend_popular_movies(
    user_id=user_id,
    train_df=train_df,
    popularity_df=popularity_rankings,
    n=10
)


show_recommendations(rec_ids, movies)

                 Model  Precision@10  Recall@10  HitRate@10
0  Popularity Baseline      0.000164   0.001639    0.001639


,movieId,title,genres
796,1041,Secrets & Lies (1996),Drama
883,1178,Paths of Glory (1957),Drama|War
1426,1949,"Man for All Seasons, A (1966)",Drama
1664,2239,Swept Away (Travolti da un insolito destino ne...,Comedy|Drama
2582,3451,Guess Who's Coming to Dinner (1967),Drama
3210,4334,Yi Yi (2000),Drama
4396,6460,"Trial, The (Procès, Le) (1962)",Drama
5773,31364,Memories of Murder (Salinui chueok) (2003),Crime|Drama|Mystery|Thriller
8301,106642,"Day of the Doctor, The (2013)",Adventure|Drama|Sci-Fi
9618,177593,"Three Billboards Outside Ebbing, Missouri (2017)",Crime|Drama


## kNN

In [3]:
from models import fit_item_knn, recommend_item_knn

user_item_matrix, item_neighbors = fit_item_knn(
    train_df=train_df,
    k_neighbors=20
)

example_user = ratings["userId"].iloc[0]

knn_rec_ids = recommend_item_knn(
    user_id=example_user,
    user_item_matrix=user_item_matrix,
    item_neighbors=item_neighbors,
    n=10
)

show_recommendations(knn_rec_ids, movies)

,movieId,title,genres
693,911,Charade (1963),Comedy|Crime|Mystery|Romance|Thriller
1960,2599,Election (1999),Comedy
2119,2816,Iron Eagle II (1988),Action|War
2511,3359,Breaking Away (1979),Comedy|Drama
2996,4011,Snatch (2000),Comedy|Crime|Thriller
3771,5267,"Rookie, The (2002)",Drama
4338,6337,Owning Mahowny (2003),Crime|Drama|Thriller
4341,6341,"Shape of Things, The (2003)",Drama
4361,6378,"Italian Job, The (2003)",Action|Crime
4795,7143,"Last Samurai, The (2003)",Action|Adventure|Drama|War


In [4]:
knn_recommend_func = lambda user_id, k: recommend_item_knn(
    user_id=user_id,
    user_item_matrix=user_item_matrix,
    item_neighbors=item_neighbors,
    n=k
)

knn_results = evaluate_recommender(
    train_df=train_df,
    test_df=test_df,
    recommend_func=knn_recommend_func,
    model_name="Item-Item kNN",
    k=10
)

knn_results

{'Model': 'Item-Item kNN',
 'Precision@10': 0.000819672131147541,
 'Recall@10': 0.00819672131147541,
 'HitRate@10': 0.00819672131147541}

### Train SVD & Recommend

In [5]:
from models import fit_svd_model, recommend_svd

svd_model = fit_svd_model(
    train_df=train_df,
    n_factors=50,
    n_epochs=20,
    lr_all=0.005,
    reg_all=0.02,
    random_state=42
)

example_user = ratings["userId"].iloc[0]

svd_rec_ids = recommend_svd(
    user_id=example_user,
    train_df=train_df,
    movies_df=movies,
    model=svd_model,
    n=10
)

show_recommendations(svd_rec_ids, movies)

,movieId,title,genres
277,318,"Shawshank Redemption, The (1994)",Crime|Drama
602,750,Dr. Strangelove or: How I Learned to Stop Worr...,Comedy|War
680,898,"Philadelphia Story, The (1940)",Comedy|Drama|Romance
690,908,North by Northwest (1959),Action|Adventure|Mystery|Romance|Thriller
733,953,It's a Wonderful Life (1946),Children|Drama|Fantasy|Romance
841,1104,"Streetcar Named Desire, A (1951)",Drama
924,1223,"Grand Day Out with Wallace and Gromit, A (1989)",Adventure|Animation|Children|Comedy|Sci-Fi
929,1228,Raging Bull (1980),Drama
933,1233,"Boot, Das (Boat, The) (1981)",Action|Drama|War
935,1235,Harold and Maude (1971),Comedy|Drama|Romance


In [7]:
svd_recommend_func = lambda user_id, k: recommend_svd(
    user_id=user_id,
    train_df=train_df,
    movies_df=movies,
    model=svd_model,
    n=k
)

svd_results = evaluate_recommender(
    train_df=train_df,
    test_df=test_df,
    recommend_func=svd_recommend_func,
    model_name="SVD (50 factors)",
    k=10
)

svd_results

{'Model': 'SVD (50 factors)',
 'Precision@10': 0.0029508196721311475,
 'Recall@10': 0.029508196721311476,
 'HitRate@10': 0.029508196721311476}

In [23]:
import random

random.seed(42)
np.random.seed(42)

def get_sampled_candidates(user_id, test_movie, train_df, movies_df, n_neg=100):
    """
    Creates candidate set:
    - 1 positive item: held-out test movie
    - n_neg negative items: random unseen movies
    """
    all_movies = set(movies_df["movieId"].unique())

    seen_movies = set(
        train_df[train_df["userId"] == user_id]["movieId"]
    )

    unseen_movies = list(all_movies - seen_movies - {test_movie})

    n_sample = min(n_neg, len(unseen_movies))
    negative_samples = random.sample(unseen_movies, n_sample)

    candidates = negative_samples + [test_movie]

    return candidates

In [24]:
def precision_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / k


def recall_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    hits = len(set(recommended_k) & set(relevant))
    return hits / len(relevant) if len(relevant) > 0 else 0


def hit_rate_at_k(recommended, relevant, k):
    recommended_k = recommended[:k]
    return int(len(set(recommended_k) & set(relevant)) > 0)

In [25]:
def recommend_popularity_sampled(user_id, train_df, popularity_df, candidates, n=10):
    """
    Popularity baseline over sampled candidate set only.
    """
    candidate_df = popularity_df[
        popularity_df["movieId"].isin(candidates)
    ].copy()

    recs = candidate_df.sort_values(
        by=["avg_rating", "rating_count"],
        ascending=False
    ).head(n)

    return recs["movieId"].tolist()


def evaluate_popularity_sampled(test_df, train_df, movies_df, popularity_df, top_n=10, n_neg=100):
    precisions = []
    recalls = []
    hits = []

    for _, row in test_df.iterrows():
        user_id = row["userId"]
        test_movie = row["movieId"]

        candidates = get_sampled_candidates(
            user_id=user_id,
            test_movie=test_movie,
            train_df=train_df,
            movies_df=movies_df,
            n_neg=n_neg
        )

        recs = recommend_popularity_sampled(
            user_id=user_id,
            train_df=train_df,
            popularity_df=popularity_df,
            candidates=candidates,
            n=top_n
        )

        relevant = [test_movie]

        precisions.append(precision_at_k(recs, relevant, top_n))
        recalls.append(recall_at_k(recs, relevant, top_n))
        hits.append(hit_rate_at_k(recs, relevant, top_n))

    return {
        "Model": "Popularity Baseline",
        f"Precision@{top_n}": np.mean(precisions),
        f"Recall@{top_n}": np.mean(recalls),
        f"HitRate@{top_n}": np.mean(hits),
        "Evaluation": f"Sampled Candidates ({n_neg} negatives)"
    }


popularity_sampled_results = evaluate_popularity_sampled(
    test_df=test_df,
    train_df=train_df,
    movies_df=movies,
    popularity_df=popularity_df,
    top_n=10,
    n_neg=100
)

popularity_sampled_results

{'Model': 'Popularity Baseline',
 'Precision@10': 0.04114754098360656,
 'Recall@10': 0.41147540983606556,
 'HitRate@10': 0.41147540983606556,
 'Evaluation': 'Sampled Candidates (100 negatives)'}

In [26]:
def recommend_item_knn_sampled(user_id, user_item_matrix, item_neighbors, candidates, n=10):
    """
    Item-item kNN recommendations restricted to sampled candidate set.
    """
    scores = score_user_items_knn(
        user_id=user_id,
        user_item_matrix=user_item_matrix,
        item_neighbors=item_neighbors
    )

    candidate_scores = scores[
        scores.index.isin(candidates)
    ]

    return candidate_scores.head(n).index.tolist()


def evaluate_item_knn_sampled(test_df, train_df, movies_df, user_item_matrix, item_neighbors, top_n=10, n_neg=100):
    precisions = []
    recalls = []
    hits = []
    skipped_users = 0

    for _, row in test_df.iterrows():
        user_id = row["userId"]
        test_movie = row["movieId"]

        if user_id not in user_item_matrix.index:
            skipped_users += 1
            continue

        candidates = get_sampled_candidates(
            user_id=user_id,
            test_movie=test_movie,
            train_df=train_df,
            movies_df=movies_df,
            n_neg=n_neg
        )

        recs = recommend_item_knn_sampled(
            user_id=user_id,
            user_item_matrix=user_item_matrix,
            item_neighbors=item_neighbors,
            candidates=candidates,
            n=top_n
        )

        relevant = [test_movie]

        precisions.append(precision_at_k(recs, relevant, top_n))
        recalls.append(recall_at_k(recs, relevant, top_n))
        hits.append(hit_rate_at_k(recs, relevant, top_n))

    return {
        "Model": "Item-Item kNN (k=20)",
        f"Precision@{top_n}": np.mean(precisions),
        f"Recall@{top_n}": np.mean(recalls),
        f"HitRate@{top_n}": np.mean(hits),
        "Skipped Users": skipped_users,
        "Evaluation": f"Sampled Candidates ({n_neg} negatives)"
    }


knn_sampled_results = evaluate_item_knn_sampled(
    test_df=test_df,
    train_df=train_df,
    movies_df=movies,
    user_item_matrix=user_item_matrix,
    item_neighbors=item_neighbors,
    top_n=10,
    n_neg=100
)

knn_sampled_results

{'Model': 'Item-Item kNN (k=20)',
 'Precision@10': 0.06245901639344261,
 'Recall@10': 0.6245901639344262,
 'HitRate@10': 0.6245901639344262,
 'Skipped Users': 0,
 'Evaluation': 'Sampled Candidates (100 negatives)'}

In [27]:
def recommend_svd_sampled(user_id, model, candidates, n=10):
    """
    SVD recommendations restricted to sampled candidate set.
    """
    predictions = []

    for movie_id in candidates:
        pred_rating = model.predict(user_id, movie_id).est
        predictions.append((movie_id, pred_rating))

    predictions.sort(key=lambda x: x[1], reverse=True)

    return [movie_id for movie_id, _ in predictions[:n]]


def evaluate_svd_sampled(test_df, train_df, movies_df, model, top_n=10, n_neg=100):
    precisions = []
    recalls = []
    hits = []

    for _, row in test_df.iterrows():
        user_id = row["userId"]
        test_movie = row["movieId"]

        candidates = get_sampled_candidates(
            user_id=user_id,
            test_movie=test_movie,
            train_df=train_df,
            movies_df=movies_df,
            n_neg=n_neg
        )

        recs = recommend_svd_sampled(
            user_id=user_id,
            model=model,
            candidates=candidates,
            n=top_n
        )

        relevant = [test_movie]

        precisions.append(precision_at_k(recs, relevant, top_n))
        recalls.append(recall_at_k(recs, relevant, top_n))
        hits.append(hit_rate_at_k(recs, relevant, top_n))

    return {
        "Model": "SVD (50 factors)",
        f"Precision@{top_n}": np.mean(precisions),
        f"Recall@{top_n}": np.mean(recalls),
        f"HitRate@{top_n}": np.mean(hits),
        "Evaluation": f"Sampled Candidates ({n_neg} negatives)"
    }


svd_sampled_results = evaluate_svd_sampled(
    test_df=test_df,
    train_df=train_df,
    movies_df=movies,
    model=svd_model,
    top_n=10,
    n_neg=100
)

svd_sampled_results

{'Model': 'SVD (50 factors)',
 'Precision@10': 0.04327868852459016,
 'Recall@10': 0.43278688524590164,
 'HitRate@10': 0.43278688524590164,
 'Evaluation': 'Sampled Candidates (100 negatives)'}

In [28]:
sampled_results_df = pd.DataFrame([
    popularity_sampled_results,
    knn_sampled_results,
    svd_sampled_results
])

sampled_results_df = sampled_results_df.drop(
    columns=["Skipped Users"],
    errors="ignore"
)

sampled_results_df

,Model,Precision@10,Recall@10,HitRate@10,Evaluation
0,Popularity Baseline,0.041148,0.411475,0.411475,Sampled Candidates (100 negatives)
1,Item-Item kNN (k=20),0.062459,0.624590,0.624590,Sampled Candidates (100 negatives)
2,SVD (50 factors),0.043279,0.432787,0.432787,Sampled Candidates (100 negatives)
